# Episode 19: Control Your Agent Costs

Multi-agent systems are powerful. They're also expensive if you're not paying attention.

**By the end of this episode, you will be able to:**
- Track token usage and cost per conversation
- Apply model tiering to cut costs without sacrificing quality
- Choose the right optimization strategy for your workload

## Why Multi-Agent Systems Are Expensive

Cost optimization is a design skill, not a compromise. The teams that build the most effective multi-agent systems are the ones that treat cost as an engineering problem from day one, not a surprise they discover in production.

A single LLM call is cheap. But multi-agent systems **multiply costs** in a way that catches people off guard:

```
Cost = N agents × M turns × tokens per turn × price per token
```

A 3-agent system with 6 rounds can cost **18x** a single LLM call. And each turn sends the full conversation history, so token counts grow quadratically.

Here's what that looks like in practice.

| Scenario | Agents | Turns | Relative Cost |
|----------|--------|-------|---------------|
| Single agent, one shot | 1 | 1 | 1x |
| Two agents, 3 rounds | 2 | 3 | ~6x |
| Three agents, 6 rounds | 3 | 6 | ~18x |
| Five agents, auto pattern | 5 | 10 | ~50x+ |

Without cost awareness, a popular agent can cost hundreds of dollars per day. The good news: most of these costs are avoidable with the right design choices.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from autogen import ConversableAgent, LLMConfig

## Part 1: Token Tracking

You can't optimize what you don't measure. Before changing anything about your system, start by tracking token usage per conversation. AG2 makes this straightforward — every result object includes cost information.

In [ ]:
llm_config = LLMConfig({"model": "gpt-4o-mini"})

agent = ConversableAgent(
    name="cost_tracker",
    system_message="You are a helpful assistant. Be concise.",
    llm_config=llm_config, human_input_mode="NEVER",
)

user = ConversableAgent(name="user", human_input_mode="NEVER")

result = user.run(agent, message="Explain the difference between TCP and UDP in two sentences.",
    max_turns=2,
)
result.process()

# Access cost information from the result
print(f"Total cost: {result.cost}")
print(f"Messages exchanged: {len(result.messages)}")
print(f"Summary: {result.summary}")

In [ ]:
# Compare cost with more turns
chatty_agent = ConversableAgent(
    name="chatty_agent",
    system_message="You are a helpful assistant. Ask a follow-up question after each answer.",
    llm_config=llm_config, human_input_mode="NEVER",
)

user = ConversableAgent(name="user", human_input_mode="NEVER")

result_chatty = user.run(chatty_agent, message="Explain the difference between TCP and UDP.",
    max_turns=4,
)

print(f"Chatty agent cost: {result_chatty.cost}")
print(f"Messages exchanged: {len(result_chatty.messages)}")
print(f"\nMore turns = more tokens = higher cost!")

## Part 2: Model Tiering

This is the single highest-ROI optimization. Use the expensive model where it matters — complex reasoning, nuanced analysis — and use cheap models everywhere else.

Researchers at Tufts University found that a multi-agent system using GPT-3.5 outperformed a single GPT-4 agent on medical reasoning tasks, at significantly lower cost. The architecture mattered more than the model size.

The pricing gap makes this a no-brainer for routing:

- **Routing/triage** — simple classification, use `gpt-4o-mini` ($0.15/1M input tokens)
- **Deep analysis** — complex reasoning, use `gpt-4o` ($2.50/1M input tokens)

That's a **16x cost difference** for a step that doesn't need the extra horsepower. Let's see it in action.

In [ ]:
from autogen.agentchat import initiate_group_chat
from autogen.agentchat.group import AgentTarget, OnCondition, StringLLMCondition, RevertToUserTarget
from autogen.agentchat.group.patterns import DefaultPattern

# Model tiering: cheap model for routing, expensive model for reasoning
fast_config = LLMConfig({"model": "gpt-4o-mini"})   # For routing
smart_config = LLMConfig({"model": "gpt-4o"})       # For reasoning

# Router uses the cheap model — it just needs to classify the query
router = ConversableAgent(
    name="router",
    system_message=(
        "You are a query router. Route complex analytical questions to the analyst. "
        "Route simple factual questions to the assistant. Be very brief."
    ),
    llm_config=fast_config,
    human_input_mode="NEVER",
)

# Analyst uses the expensive model — it needs deep reasoning
analyst = ConversableAgent(
    name="analyst",
    system_message=(
        "You are a senior analyst. Provide deep, thoughtful analysis "
        "with pros, cons, and recommendations. Be thorough but concise."
    ),
    llm_config=smart_config,
    human_input_mode="NEVER",
)

# Simple assistant uses the cheap model too
assistant = ConversableAgent(
    name="assistant",
    system_message="You answer simple factual questions. Be concise.",
    llm_config=fast_config,
    human_input_mode="NEVER",
)

user = ConversableAgent(name="user", human_input_mode="NEVER")

router.handoffs.add_llm_conditions([
    OnCondition(
        target=AgentTarget(analyst),
        condition=StringLLMCondition(prompt="When the question requires complex analysis or strategic thinking."),
    ),
    OnCondition(
        target=AgentTarget(assistant),
        condition=StringLLMCondition(prompt="When the question is a simple factual query."),
    ),
])
analyst.handoffs.set_after_work(RevertToUserTarget())
assistant.handoffs.set_after_work(RevertToUserTarget())

pattern = DefaultPattern(
    initial_agent=router,
    agents=[router, analyst, assistant],
    user_agent=user,
    group_manager_args={"llm_config": fast_config},
)

print("Router uses: gpt-4o-mini (cheap)")
print("Analyst uses: gpt-4o (smart)")
print("Assistant uses: gpt-4o-mini (cheap)")

In [ ]:
# Simple question — routed to cheap assistant
result_simple, ctx_simple, last_simple = initiate_group_chat(
    pattern=pattern,
    messages="What is the capital of Japan?",
    max_rounds=4,
)
print("--- Simple Question ---")
print(f"Routed to: {last_simple.name}")
print(f"Cost: {result_simple.cost if hasattr(result_simple, 'cost') else 'N/A'}")

In [ ]:
# Complex question — routed to expensive analyst
result_complex, ctx_complex, last_complex = initiate_group_chat(
    pattern=pattern,
    messages="Should our startup use a microservices or monolithic architecture? We have 5 engineers and expect rapid growth.",
    max_rounds=4,
)
print("--- Complex Question ---")
print(f"Routed to: {last_complex.name}")
print(f"Cost: {result_complex.cost if hasattr(result_complex, 'cost') else 'N/A'}")

## The Counter-Intuitive Math: Multi-Agent Cheap Can Beat Single Expensive

Here's what surprises most people: a multi-agent system using a cheap model can be **both better AND cheaper** than a single call to an expensive model.

The reason is that agents have more predictable, focused workloads. A single GPT-4o call trying to research, analyze, AND write a report needs the full reasoning capacity of the expensive model. Three focused agents — each doing one simple task — can get by with the much cheaper GPT-4o-mini.

Let's do the math with real pricing (March 2026):

| Approach | Model | Input Tokens | Output Tokens | Total Cost |
|----------|-------|-------------|--------------|------------|
| **Single GPT-4o call** | gpt-4o | ~2,000 | ~1,000 | **$0.0075** |
| **3 agents × 2 rounds, GPT-4o-mini** | gpt-4o-mini | ~6,000 | ~3,000 | **$0.0014** |

The multi-agent approach uses **3× more tokens** but costs **5× less** — because gpt-4o-mini is 16× cheaper per token than gpt-4o.

And here's the kicker: the Tufts University team found the multi-agent approach actually produced **better results** because each agent could focus on its specific task without juggling multiple responsibilities in a single prompt.

This is the key insight: **the right architecture with a cheap model beats the wrong architecture with an expensive model.** Design your agent system first, then pick the cheapest model that can handle each role.

## Part 3: Optimization Strategies

Model tiering is the biggest lever, but it's not the only one. Here are five practical strategies that compound with each other.

### 1. Reduce `max_turns`

Fewer turns means fewer tokens. Most tasks can be solved in 2-3 turns — not 10. Setting a generous turn limit "just in case" is one of the most common sources of unnecessary spend.

```python
# Instead of
result = agent.run(message=query, max_turns=10)  # Expensive!

# Use
result = agent.run(message=query, max_turns=3)   # Usually sufficient
```

### 2. Shorter system messages

System messages are sent with **every turn**. A 500-token system message in a 6-turn conversation costs 3,000 tokens just for instructions. Keep them tight.

```python
# Instead of
system_message="You are a highly experienced senior customer service representative "
               "who specializes in handling complex billing inquiries with empathy "
               "and precision..."  # 50+ words

# Use
system_message="Handle billing questions. Be concise and accurate."  # 7 words
```

### 3. Choose simpler patterns

Not every task needs the Auto pattern with a manager. If your workflow is predictable, RoundRobin or Handoffs will get the job done at a fraction of the cost.

### 4. Cache frequent responses

If users ask the same questions repeatedly, cache the responses instead of calling the LLM again. This is especially effective for FAQ-style agents.

### 5. Batch processing

Process multiple queries in a single agent session instead of creating a new session for each. You avoid re-sending system messages and establishing context from scratch.

### Cost Comparison by Pattern

Different orchestration patterns have very different cost profiles. This table will help you pick the right one for your budget.

| Pattern | LLM Calls per Round | Relative Cost | When to Use |
|---------|--------------------|--------------|---------------|
| **Single Agent** | 1 | Lowest | Simple tasks |
| **RoundRobin** | 1 (agent only) | Low | Predictable workflows |
| **Handoffs** | 1-2 (route + handle) | Low-Medium | Specialized routing |
| **Auto** | 2 (manager + agent) | Medium | Dynamic collaboration |
| **Redundant** | N (all agents) | High | When you need consensus |

The pattern you choose often matters more than the model you choose. A well-designed handoff system with `gpt-4o-mini` will usually beat a single `gpt-4o` agent — and cost less too.

In [ ]:
# Demonstrate: same task, different turn limits, different costs
llm_config = LLMConfig({"model": "gpt-4o-mini"})

query = "List 3 benefits of cloud computing."

# Run with 2 turns (efficient)
efficient_agent = ConversableAgent(
    name="efficient",
    system_message="Be concise. Answer in a bullet list.",
    llm_config=llm_config, human_input_mode="NEVER",
)

user = ConversableAgent(name="user", human_input_mode="NEVER")

result_2 = user.run(efficient_agent, message=query, max_turns=2)
print(f"2 turns - Cost: {result_2.cost}, Messages: {len(result_2.messages)}")

# Run with 6 turns (wasteful for this task)
verbose_agent = ConversableAgent(
    name="verbose",
    system_message="Answer questions. Then elaborate and ask if the user wants more detail.",
    llm_config=llm_config, human_input_mode="NEVER",
)

result_6 = user.run(verbose_agent, message=query, max_turns=6)
print(f"6 turns - Cost: {result_6.cost}, Messages: {len(result_6.messages)}")

## Using Local Models for Zero-Cost Agents

For development, testing, or cost-insensitive agents, local models via [Ollama](https://ollama.ai/) eliminate API costs entirely. This is a great way to iterate quickly without watching your bill climb.

```python
# Free, runs locally
local_config = LLMConfig({"model": "llama3.1"})

with local_config:
    local_agent = ConversableAgent(
        name="local_agent",
        system_message="You are a helpful assistant.",
        human_input_mode="NEVER",
    )
```

Local models work well for development and testing (unlimited free calls), simple routing and triage (where quality is less critical), and privacy-sensitive tasks where data stays on your machine. Reserve cloud models for production workloads that need high quality, complex reasoning, or the latest capabilities.

## Budget Limits and Cost Alerts

In production, hope is not a strategy. Set hard limits to prevent runaway costs.

```python
class BudgetTracker:
    def __init__(self, max_budget: float):
        self.max_budget = max_budget
        self.total_spent = 0.0

    def check_budget(self, estimated_cost: float) -> bool:
        if self.total_spent + estimated_cost > self.max_budget:
            print(f"Budget exceeded! Spent: ${self.total_spent:.4f}, Limit: ${self.max_budget:.4f}")
            return False
        return True

    def record_cost(self, cost: float):
        self.total_spent += cost
        remaining = self.max_budget - self.total_spent
        if remaining < self.max_budget * 0.1:  # 10% remaining
            print(f"WARNING: Only ${remaining:.4f} budget remaining!")

# Set a $1 budget for the session
budget = BudgetTracker(max_budget=1.00)
```

A simple tracker like this catches problems before they become expensive. In a real system, you'd wire this into your monitoring and alerting infrastructure.

## When to Use Fewer vs More Agents

More agents isn't always better. The right number depends on the shape of your problem.

| Question | Fewer Agents | More Agents |
|----------|-------------|-------------|
| How distinct are the subtasks? | Overlapping | Clearly separate |
| How complex is each subtask? | Simple | Requires specialization |
| How important is cost? | Critical | Secondary |
| What's the latency budget? | Tight (<2s) | Flexible (>5s) |
| How often is this called? | High volume | Low volume |

The rule of thumb: start with the fewest agents that solve the problem. Add more only when a single agent demonstrably can't handle the task. Every agent you add multiplies cost, latency, and debugging complexity.

## Try It Yourself

Add model tiering to your Episode 10 customer service system:

1. Use `gpt-4o-mini` for the triage agent (cheap routing)
2. Use `gpt-4o` for the specialist agents (quality answers)
3. Run the same 3 queries with both configurations
4. Compare total costs

In [ ]:
# Your code here — add model tiering and compare costs

## What's Next

You know how to build multi-agent systems and how to keep them affordable. In **Episode 20**, you'll take the next step: **deploying your agent to the cloud** so real users can interact with it.